# Gate `torch.compile` with TensorGuard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/02_torch_compile.ipynb)

`guarded_compile` verifies a module and only then hands it to `torch.compile`, so a verified model is compiled while a shape bug is caught *before* a single graph is built.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
import torch, torch.nn as nn
from tensorguard import guarded_compile

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(32, 16)
        self.b = nn.Linear(16, 4)
    def forward(self, x):
        return self.b(torch.relu(self.a(x)))

compiled = guarded_compile(Net(), input_shapes={'x': ('batch', 32)})
print('compiled output shape:', tuple(compiled(torch.randn(8, 32)).shape))

The same shape contract, checked statically, rejects a buggy net:

In [ ]:
from tensorguard import verify_architecture
bad = '''
import torch, torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(32, 16)
        self.b = nn.Linear(15, 4)
    def forward(self, x):
        return self.b(torch.relu(self.a(x)))
'''
r = verify_architecture(bad, input_shapes={'x': ('batch', 32)})
print('buggy net:', r.status, '->', r.bugs[0].message)
assert r.status == 'UNSAFE'